# `ci-monitoring-simulation` — 2-minute quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lesiayanytska78/ci-monitoring-simulation/blob/main/notebooks/quickstart.ipynb)

This notebook reproduces the headline result of the project in your browser, with **no setup**.

**The claim, in one line:** a conventional adaptive-baseline energy-anomaly detector has an *inertia blind spot* — a slow, creeping fault drifts the baseline along with it and is never flagged. A **fixed, event-anchored reference + residual-CUSUM** detector holds the baseline at the start of the shift and catches the same fault.

Below, we simulate one day of a machining work center, inject a slow-ramp spindle fault, and run **both** detectors on the identical signal. The deployed detector misses it; the anchored detector catches it.

Run each cell with **Shift+Enter** (or *Runtime ▸ Run all*).

## 1 · Install the package from PyPI

Running in **Colab**? Nothing to set up — the cell below installs everything for you. Just run it.

Running this notebook **locally** instead? You need **Python 3.9 or newer** with `pip` (check yours with `python3 --version`). The `%pip` line below installs the package and its dependencies (numpy, pandas, matplotlib) automatically. Full local-setup notes are in the [README](https://github.com/lesiayanytska78/ci-monitoring-simulation#requirements-for-local-use).

In [ ]:
%pip install -q ci-monitoring-simulation

## 2 · Simulate one day of a machining work center

In [ ]:
import cimonitoring as ci
print("cimonitoring version:", ci.__version__)

# A calibrated 24-hour energy substrate (1 Hz) for a single work center
substrate = ci.simulate_work_center(ci.Config(seed=1))
substrate[["t_s", "state", "total_kw"]].head()

## 3 · Inject a slow-ramp fault

A creeping +2 kW spindle anomaly that ramps in over a full hour starting at hour 12 — the kind of gradual drift (worn tooling, degrading bearing) that fools an adaptive baseline.

In [ ]:
specs = [ci.AnomalySpec(
    onset_hour=12, duration_minutes=240, magnitude_kw=2.0,
    onset_profile="ramp", onset_ramp_seconds=3600,   # 1-hour ramp-in
    affects="spindle", label="slow ramp")]

substrate = ci.inject_anomalies(substrate, ci.AnomalyConfig(specs))
emission_factor = ci.CarbonConfig().static_emission_factor_kg_per_kwh

## 4 · Run both detectors on the identical signal

`run_monitoring` is the conventional adaptive-baseline detector. `run_monitoring_anchored` is the proposed event-anchored + residual-CUSUM detector.

In [ ]:
deployed = ci.run_monitoring(substrate, ci.MonitorConfig(), emission_factor)
anchored = ci.run_monitoring_anchored(
    substrate, ci.AnchoredMonitorConfig(detector="anchored_cusum"), emission_factor)

ev_dep = ci.evaluate(deployed, substrate, specs)["per_fault"][0]
ev_anc = ci.evaluate(anchored, substrate, specs)["per_fault"][0]

def line(name, ev):
    if ev["warning_detected"]:
        return f"{name:>9}:  DETECTED  (warning at {ev['warning_latency_min']:.0f} min, attribution={ev['attribution']})"
    return f"{name:>9}:  MISSED    (never alerts)"

print(line("deployed", ev_dep))
print(line("anchored", ev_anc))

## 5 · Visualize it

In [ ]:
import matplotlib.pyplot as plt

def first_alert_hour(df, level):
    hit = df[df["alert_level"] >= level]
    return hit["t_s"].iloc[0] / 3600 if len(hit) else None

h = anchored["t_s"] / 3600
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(h, anchored["ci_per_piece_kg_obs"], lw=1.0, label="observed CI per piece")
ax.plot(h, anchored["threshold_kg_per_piece"], lw=1.0, ls="--", color="grey",
        label="anchored threshold")
ax.axvspan(12, 16, color="orange", alpha=0.12, label="fault active (truth)")

wa = first_alert_hour(anchored, 1)
cr = first_alert_hour(anchored, 2)
if wa: ax.axvline(wa, color="goldenrod", ls=":", label=f"anchored WARNING ({(wa-12)*60:.0f} min)")
if cr: ax.axvline(cr, color="red", ls=":", label=f"anchored CRITICAL ({(cr-12)*60:.0f} min)")

dep_w = first_alert_hour(deployed, 1)
ax.set_title("Slow-ramp fault: deployed detector "
             + ("alerts" if dep_w else "never alerts")
             + ", anchored detector catches it")
ax.set_xlabel("hour of day")
ax.set_ylabel("CI per piece (kg CO2e)")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

## What you just saw

On this identical signal, the conventional adaptive-baseline detector **never alerts** — the slow ramp drifts its baseline along with the fault. The event-anchored + residual-CUSUM detector flags a **warning at ~47 min** and **critical at ~56 min**, with correct spindle attribution and zero false positives.

This is one seed for intuition; the paper establishes the effect across 200-seed comparisons with confidence intervals, a detector ablation, multiple severities, and validation on **real** CNC spindle-power traces.

### Learn more

- **Repository:** https://github.com/lesiayanytska78/ci-monitoring-simulation
- **Archived release (DOI):** https://doi.org/10.5281/zenodo.20634187
- **PyPI:** https://pypi.org/project/ci-monitoring-simulation/

### Cite

If you use this in your work, please cite the archived software (see `CITATION.cff` in the repository) and the paper.